In [49]:
from classy import Class
import numpy as np
# from cosmoprimo import Cosmology
import cosmoprimo

In [18]:
default_cosmo = {'A_s': 2.10732e-9,\
                 'n_s': 0.96824,\
                 'alpha_s': 0.,\
                 'h': 0.6770,\
                 'N_ur': 1.0196,\
                 'N_ncdm': 2,\
                 'm_ncdm': '0.01,0.05',\
                 'tau_reio': 0.0568,\
                 'omega_b': 0.02247,\
                 'omega_cdm': 0.11923,\
                 'Omega_k': 0.,\
                 'output': 'mPk',\
                 'z_max_pk': 2.}
                #  'P_k_max_h/Mpc': 2.,\
                #  'z_pk': '0.0,6'}

In [53]:
cosmo = cosmoprimo.fiducial.DESI(engine='camb')

In [ ]:
ba = cosmo.get_background()

z_arr = np.linspace(0.01, 2.0, 100)
f = 
f_arr = ba.growth_rate(z_arr)   # f = d ln D / d ln a
D_arr = ba.growth_factor(z_arr) # D(z), normalized to 1 at z=0

AttributeError: 'Background' object has no attribute 'growth_rate'

In [56]:
print([m for m in dir(ba) if 'growth' in m.lower()])
fo = cosmo.get_fourier()
print([m for m in dir(fo) if 'growth' in m.lower()])

[]
[]


In [70]:
import matplotlib.pyplot as plt

In [84]:
# display DESI fid parameters
# cosmo.A_s
Ob = cosmo.Omega0_b
Ocdm = cosmo.Omega0_cdm
h = cosmo.h
obh2 = Ob * h**2
ocdmh2 = Ocdm * h**2
print(f"DESI fiducial parameters: Omega_b={Ob:.4f}, Omega_cdm={Ocdm:.4f}, h={h:.4f}, Omega_b h^2={obh2:.5f}, Omega_cdm h^2={ocdmh2:.5f}")


DESI fiducial parameters: Omega_b=0.0493, Omega_cdm=0.2645, h=0.6736, Omega_b h^2=0.02237, Omega_cdm h^2=0.12000


In [ ]:
# z = [0.3, 0.51, 0.71, 0.92, 0.95, 0.93, 1.32, 1.49]
# P0 = [9.2e3, 8.9e3, 8.9e3, 8.4e3, 2.6e3, 5.9e3, 2.9e3, 5e3]

z = [0.3, 0.51, 0.71, 0.92, 0.95, 1.32, 1.49]
P0 = [9.2e3, 8.9e3, 8.9e3, 8.4e3, 2.6e3, 2.9e3, 5e3]

In [117]:
import camb
import numpy as np

pars = camb.CAMBparams()
pars.set_cosmology(H0=67.36, ombh2=0.02237, omch2=0.1200)
pars.InitPower.set_params(As=2.083e-9, ns=0.9649)
pars.set_matter_power(redshifts=list(z), kmax=10.0)

results = camb.get_results(pars)

fsigma8 = results.get_fsigma8()   # shape (n_redshifts,)
sigma8  = results.get_sigma8()    # shape (n_redshifts,)
f       = fsigma8 / sigma8        # f(z)
f = f[::-1]  # reverse to match z_arr order

Note: redshifts have been re-sorted (earliest first)


In [129]:
print(f)

[0.68587588 0.7655918  0.82178204 0.86495008 0.87006032 0.91733803
 0.93186466]


In [120]:
# plt.plot(z[::-1], fsigma8, label='CAMB growth rate f')
# plt.axvline(0.1, color='k', ls='--', lw=0.5)
# plt.axvline(1.7, color='k', ls='--', lw=0.5)
# plt.ylim(0.18, 0.65)

In [ ]:
PK = results.get_matter_power_interpolator(
        nonlinear=False,       # True for halofit
        var1='delta_tot',
        var2='delta_tot',
        hubble_units=True,    # k in h/Mpc, P in (Mpc/h)^3
        k_hunit=True
)

BGS 0.1 − 0.4 300,017 0.30 ∼ 9.2 × 103 1.7
LRG1 0.4 − 0.6 506,905 0.51 ∼ 8.9 × 103 2.6
LRG2 0.6 − 0.8 771,875 0.71 ∼ 8.9 × 103 4.0

LRG3 0.8 − 1.1 859,824 0.92 ∼ 8.4 × 103 5.0
ELG1 0.8 − 1.1 1,016,340 0.95 ∼ 2.6 × 103 2.0
LRG3+ELG1 0.8 − 1.1 1,876,164 0.93 ∼ 5.9 × 103 6.5

ELG2 1.1 − 1.6 1,415,687 1.32 ∼ 2.9 × 103 2.7
QSO 0.8 − 2.1 856,652 1.49 ∼ 5.0 × 103 1.5


In [ ]:
# z = [0.3, 0.51, 0.71, 0.92, 0.95, 0.93, 1.32, 1.49]
# P0 = [9.2e3, 8.9e3, 8.9e3, 8.4e3, 2.6e3, 5.9e3, 2.9e3, 5e3]

z = [0.3, 0.51, 0.71, 0.92, 0.95, 1.32, 1.49] # removed LRG3+ELG1
P0 = [9.2e3, 8.9e3, 8.9e3, 8.4e3, 2.6e3, 2.9e3, 5e3]

In [ ]:
# k = np.logspace(-4, 1, 500)  # k in h/Mpc
z = np.array(z)

In [132]:
# compute matter power spectrum
Pk = PK.P(z, 0.14)  

In [138]:
u = [(2/3) * f_i for f_i in f]
# t = [(f[i]**2)/5 - P0[i]/(2*Pk[i]) for i in range(len(P0))] # corrected factor 2 from normalization
t = [(f[i]**2)/5 - P0[i]/Pk[i] for i in range(len(P0))]
b = []

for i in range(len(P0)):
    delta = np.sqrt(u[i]**2 - 4 * t[i])
    b_i = [(-u[i] + delta) / 2, (-u[i] - delta) / 2]
    b.append(b_i)

In [139]:
for i in range(len(b)):
    if b[i][0] > 0:
        print(f"z = {z[i]:.2f}, b = {b[i][0]:.4f}")

z = 0.30, b = 1.6405
z = 0.51, b = 1.7932
z = 0.71, b = 1.9870
z = 0.92, b = 2.1333
z = 0.95, b = 1.0584
z = 1.32, b = 1.3651
z = 1.49, b = 2.0458
